Create a machine learning model to predict if a bank client is about to cancel the account.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.utils import shuffle


In [2]:
def upsample(features, target):
    '''upsample df by multiplying it where target is 1'''
    # number to multiply the minority class
    multiplier = round(target.value_counts()[0] /
                       target.value_counts()[1])
    
    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_up = pd.concat([features_zero] + [features_one] * multiplier)
    target_up = pd.concat([target_zero] + [target_one] * multiplier)

    features_up, target_up = shuffle(features_up, target_up, random_state=123)

    return features_up, target_up

def downsample(features, target):
    '''downsample df by sampling a fraction where target is 0'''
    fraction = target.value_counts()[1] / target.value_counts()[0]

    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_down = pd.concat(
        [features_zero.sample(frac=fraction, random_state=123)] + 
        [features_one]
    )
    target_down = pd.concat(
        [target_zero.sample(frac=fraction, random_state=123)] +
        [target_one]
    )

    features_down, target_down = shuffle(
        features_down,
        target_down,
        random_state=123
    )

    return features_down, target_down     

In [3]:
df = pd.read_csv('./data/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [7]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

There are null values in the 'Tenure' column. Verify if they are 0.

In [8]:
df['Tenure'].value_counts(dropna=False)

Tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
NaN     909
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

These null are not 0, because there are observations with 0. Verify if there is something in common with these customers.

In [9]:
df[df['Tenure'].isnull()].sample(10, random_state=123)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7811,611,France,Male,30,NaN,104145.65,1,0,0,159629.64,0
3543,675,France,Male,49,NaN,135133.39,1,0,1,179521.24,1
6471,850,France,Female,40,NaN,0.00,2,1,0,35034.15,0
94,730,Spain,Male,42,NaN,0.00,2,0,1,85982.47,0
6187,704,France,Male,36,NaN,114370.41,1,0,1,66810.48,0
7026,677,France,Female,27,NaN,0.00,2,0,1,114685.92,0
5314,743,Germany,Male,45,NaN,144677.19,3,1,0,22512.44,1
4445,739,France,Male,29,NaN,0.00,2,1,1,164484.78,0
9006,661,Spain,Male,20,NaN,0.00,1,1,0,110252.53,0
1960,655,Spain,Female,36,NaN,135515.76,1,1,0,86013.96,0


There is nothing in common between them. These will be discarded.

In [10]:
df = df.dropna()

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [11]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

### 2.2 - Encode Cathegorical Characteristics

In [12]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [13]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Some models performs better with One-Hot Encoding, others with Label Encoding.

Two dfs will be created, one for each encoding technique.

#### 2.2.1 One-Hot Encoding

In [14]:
df_ohe = pd.get_dummies(df, drop_first=True)
df_ohe.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


#### 2.2.2 Label Encoding

In [15]:
cat_columns = ['Geography', 'Gender']

encoder = OrdinalEncoder()
df_cat = pd.DataFrame(
    encoder.fit_transform(df[cat_columns]), 
    columns=cat_columns
)

df_le = df
df_le[cat_columns] = df_cat
df_le.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0.0,0.0,42,2.0,0.00,1,1,1,101348.88,1
1,608,2.0,0.0,41,1.0,83807.86,1,0,1,112542.58,0
2,502,0.0,0.0,42,8.0,159660.80,3,1,0,113931.57,1
3,699,0.0,0.0,39,1.0,0.00,2,0,0,93826.63,0
4,850,2.0,0.0,43,2.0,125510.82,1,1,1,79084.10,0


### 2.3 - Splitting

In [16]:
# split df_ohe into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_ohe, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_ohe = df_train.drop('Exited', axis=1)
target_train_ohe = df_train['Exited']

features_val_ohe = df_val.drop('Exited', axis=1)
target_val_ohe = df_val['Exited']

features_test_ohe = df_test.drop('Exited', axis=1)
target_test_ohe = df_test['Exited']

In [17]:
# split df_le into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_le, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_le = df_train.drop('Exited', axis=1)
target_train_le = df_train['Exited']

features_val_le = df_val.drop('Exited', axis=1)
target_val_le = df_val['Exited']

features_test_le = df_test.drop('Exited', axis=1)
target_test_le = df_test['Exited']

### 2.4 - Scaling

In [18]:
# columns to scale
cols_sca = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
            'EstimatedSalary']

# scale df_ohe
scaler_ohe = StandardScaler()
scaler_ohe.fit(features_train_ohe[cols_sca])

features_train_ohe[cols_sca] = scaler_ohe.transform(features_train_ohe[cols_sca])
features_val_ohe[cols_sca] = scaler_ohe.transform(features_val_ohe[cols_sca])
features_test_ohe[cols_sca] = scaler_ohe.transform(features_test_ohe[cols_sca])

# scale df_le
scaler_le = StandardScaler()
scaler_le.fit(features_train_le[cols_sca])

features_train_le[cols_sca] = scaler_le.transform(features_train_le[cols_sca])
features_val_le[cols_sca] = scaler_le.transform(features_val_le[cols_sca])
features_test_le[cols_sca] = scaler_le.transform(features_test_le[cols_sca])

## 3 - Model with Imbalance
Model without correcting the target imbalance.

### 3.1 - LogisticRegression
Logistic Regression model performs better with One-hot encoding.

In [19]:
model_lr = LogisticRegression(solver='liblinear', random_state=123)
model_lr.fit(features_train_ohe, target_train_ohe)

score_train = model_lr.score(features_train_ohe, target_train_ohe)
score_val = model_lr.score(features_val_ohe, target_val_ohe)

print(
    f'train score: {score_train:.3f}\n'
    f'val score: {score_val:.3f}'
)

train score: 0.809
val score: 0.822


In [20]:
predicted_valid_lr = model_lr.predict(features_val_ohe)
f1_lr = f1_score(predicted_valid_lr, target_val_ohe)
print(f'F1 Score for Logistic Regression: {f1_lr:.3f}')

F1 Score for Logistic Regression: 0.328


### 3.2 - RandomForestClassifier

Random Forest model performs better with Label Encoding.

In [21]:
best_score_train = 0
best_score_val = 0
best_n_estimators = 0
best_depth = 0

for estimators in range(1, 31):
    # print(f'est: {estimators}')
    for depth in range(1, 21):
        # print(f'depth: {depth}')
        model = RandomForestClassifier(n_estimators=estimators,
                                       max_depth=depth,
                                       random_state=123)
        model.fit(features_train_le, target_train_le)

        score_train = model.score(features_train_le, target_train_le)
        score_val = model.score(features_val_le, target_val_le)

        if score_val > best_score_val:
            best_score_val = score_val
            best_score_train = score_train
            best_n_estimators = estimators
            best_depth = depth

model_rfc = RandomForestClassifier(n_estimators=best_n_estimators,
                               max_depth=best_depth,
                               random_state=123)
model_rfc.fit(features_train_le, target_train_le)

print(
    f'best_depth: {best_depth}\n'
    f'best_n_estimators: {best_n_estimators}\n'
    f'train score: {best_score_train:.3f}\n'
    f'val score: {best_score_val:.3f}'
)

best_depth: 14
best_n_estimators: 24
train score: 0.959
val score: 0.883


In [22]:
predicted_valid_rfc = model_rfc.predict(features_val_le)
f1_rfc = f1_score(predicted_valid_rfc, target_val_le)
print(f'F1 Score for Random Forest Classifier: {f1_rfc:.3f}')

F1 Score for Random Forest Classifier: 0.623


### 3.3 - DecisionTreeClassifier

In [23]:
best_score_train = 0
best_score_val = 0
best_depth = 0

for depth in range(1, 51):
    # print(f'depth: {depth}')
    model = DecisionTreeClassifier(max_depth=depth, random_state=123)
    model.fit(features_train_le, target_train_le)

    score_train = model.score(features_train_le, target_train_le)
    score_val = model.score(features_val_le, target_val_le)

    if score_val > best_score_val:
        best_score_val = score_val
        best_score_train = score_train
        best_depth = depth

model_dtc = DecisionTreeClassifier(max_depth=best_depth, random_state=123)
model_dtc.fit(features_train_le, target_train_le)

print(
    f'best_depth: {best_depth}\n'
    f'train score: {best_score_train:.3f}\n'
    f'val score: {best_score_val:.3f}'
)


best_depth: 5
train score: 0.859
val score: 0.875


In [24]:
predicted_valid_dtc = model_dtc.predict(features_val_le)
f1_dtc = f1_score(predicted_valid_dtc, target_val_le)
print(f'F1 Score for Decision Tree Classifier: {f1_dtc:.3f}')

F1 Score for Decision Tree Classifier: 0.576


## 4 - Model without Imbalance
Model correcting the target imbalance.

In [35]:
# one-hot encoding with upsample
features_train_ohe_up, target_train_ohe_up = upsample(features_train_ohe, target_train_ohe)

# label encoding with upsample
features_train_le_up, target_train_le_up = upsample(features_train_le, target_train_le)

# one-hot encoding with downsample
features_train_ohe_down, target_train_ohe_down = downsample(features_train_ohe, target_train_ohe)

# label encoding with downsample
features_train_le_down, target_train_le_down = downsample(features_train_le, target_train_le)

### 4.1 LogisticRegression


#### 4.1.1 - Model Class Weight Parameter
Correcting with model class_weight='balanced parameter, that changes the weight to compensate the imbalance.

In [26]:
model_lr_we = LogisticRegression(solver='liblinear', 
                                random_state=123, 
                                class_weight='balanced')
model_lr_we.fit(features_train_ohe, target_train_ohe)

score_train = model_lr_we.score(features_train_ohe, target_train_ohe)
score_val = model_lr_we.score(features_val_ohe, target_val_ohe)

print(
    f'train score: {score_train:.3f}\n'
    f'val score: {score_val:.3f}'
)

train score: 0.713
val score: 0.719


In [27]:
predicted_valid_lr_we = model_lr_we.predict(features_val_ohe)
f1_lr_we = f1_score(predicted_valid_lr_we, target_val_ohe)
print(f'F1 Score for Logistic Regression: {f1_lr_we:.3f}')

F1 Score for Logistic Regression: 0.481


#### 4.1.2 - Upsample

In [28]:
model_lr_up = LogisticRegression(solver='liblinear', random_state=123)
model_lr_up.fit(features_train_ohe_up, target_train_ohe_up)

score_train = model_lr_up.score(features_train_ohe_up, target_train_ohe_up)
score_val = model_lr_up.score(features_val_ohe, target_val_ohe)

print(
    f'train score: {score_train:.3f}\n'
    f'val score: {score_val:.3f}'
)

train score: 0.714
val score: 0.714


In [29]:
predicted_valid_lr_up = model_lr_up.predict(features_val_ohe)
f1_lr_up = f1_score(predicted_valid_lr_up, target_val_ohe)
print(f'F1 Score for Logistic Regression Upsampled: {f1_lr_up:.3f}')

F1 Score for Logistic Regression Upsampled: 0.480


#### 4.1.3 - Downsample

In [43]:
model_lr_down = LogisticRegression(solver='liblinear', random_state=123)
model_lr_down.fit(features_train_ohe_down, target_train_ohe_down)

score_train = model_lr_down.score(features_train_ohe_down, target_train_ohe_down)
score_val = model_lr_down.score(features_val_ohe, target_val_ohe)

print(
    f'train score: {score_train:.3f}\n'
    f'val score: {score_val:.3f}'
)

train score: 0.713
val score: 0.722


In [44]:
predicted_valid_lr_down = model_lr_down.predict(features_val_ohe)
f1_lr_down = f1_score(predicted_valid_lr_down, target_val_ohe)
print(f'F1 Score for Logistic Regression Downsampled: {f1_lr_down:.3f}')

F1 Score for Logistic Regression Downsampled: 0.485


### 4.2 - RandomForestClassifier

#### 4.2.1 - Model Class Weight Parameter

#### 4.2.2 - Upsample

In [30]:
best_score_train = 0
best_score_val = 0
best_n_estimators = 0
best_depth = 0

for estimators in range(1, 31):
    # print(f'est: {estimators}')
    for depth in range(1, 21):
        # print(f'depth: {depth}')
        model = RandomForestClassifier(n_estimators=estimators,
                                       max_depth=depth,
                                       random_state=123)
        model.fit(features_train_le_up, target_train_le_up)

        score_train = model.score(features_train_le_up, target_train_le_up)
        score_val = model.score(features_val_le, target_val_le)

        if score_val > best_score_val:
            best_score_val = score_val
            best_score_train = score_train
            best_n_estimators = estimators
            best_depth = depth

model_rfc_up = RandomForestClassifier(n_estimators=best_n_estimators,
                                      max_depth=best_depth,
                                      random_state=123)
model_rfc_up.fit(features_train_le_up, target_train_le_up)

print(
    f'best_depth: {best_depth}\n'
    f'best_n_estimators: {best_n_estimators}\n'
    f'train score: {best_score_train:.3f}\n'
    f'val score: {best_score_val:.3f}'
)

best_depth: 19
best_n_estimators: 24
train score: 1.000
val score: 0.861


In [31]:
predicted_valid_rfc_up = model_rfc_up.predict(features_val_le)
f1_rfc_up = f1_score(predicted_valid_rfc_up, target_val_le)
print(f'F1 Score for Random Forest Classifier upsampled: {f1_rfc_up:.3f}')

F1 Score for Random Forest Classifier upsampled: 0.603


#### 4.2.3 - Downsample

### 4.3 - DecisionTreeClassifier

#### 4.3.1 - Model Class Weight Parameter

#### 4.3.2 - Upsample

In [32]:
best_score_train = 0
best_score_val = 0
best_depth = 0

for depth in range(1, 51):
    # print(f'depth: {depth}')
    model = DecisionTreeClassifier(max_depth=depth, random_state=123)
    model.fit(features_train_le_up, target_train_le_up)

    score_train = model.score(features_train_le_up, target_train_le_up)
    score_val = model.score(features_val_le, target_val_le)

    if score_val > best_score_val:
        best_score_val = score_val
        best_score_train = score_train
        best_depth = depth

model_dtc_up = DecisionTreeClassifier(max_depth=best_depth, random_state=123)
model_dtc_up.fit(features_train_le_up, target_train_le_up)

print(
    f'best_depth: {best_depth}\n'
    f'train score: {best_score_train:.3f}\n'
    f'val score: {best_score_val:.3f}'
)

best_depth: 5
train score: 0.753
val score: 0.819


In [33]:
predicted_valid_dtc_up = model_dtc_up.predict(features_val_le)
f1_dtc_up = f1_score(predicted_valid_dtc_up, target_val_le)
print(f'F1 Score for Decision Tree Classifier upsampled: {f1_dtc_up:.3f}')

F1 Score for Decision Tree Classifier upsampled: 0.596


#### 4.3.3 - Downsample

## 5 - F1 Scores

In [34]:
var_names = %who_ls
for var in var_names:
    if var.startswith('f1') and var != 'f1_score':
        value = globals()[var]
        print(f'{var} = {value:.3f}')

f1_dtc = 0.576
f1_dtc_up = 0.596
f1_lr = 0.328
f1_lr_up = 0.480
f1_lr_we = 0.481
f1_rfc = 0.623
f1_rfc_up = 0.603
